In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
df = pd.read_csv(r'rd/Final_SDM_Data.csv')

In [3]:
# 1. Automate feature selection
non_predictors = ['presence', 'koppen_zone', 'decimalLongitude', 'decimalLatitude']
predictor_cols = [col for col in df.columns if col not in non_predictors]

X = df[predictor_cols]
y = df['presence']

# 2. Define the correlation filtering function
def drop_highly_correlated(data, threshold=0.5):
    data = data.copy()

    while True:
        corr = data.corr().abs()
        upper = corr.where(
            np.triu(np.ones(corr.shape), k=1).astype(bool)
        )
        max_corr = upper.max().max()

        if max_corr < threshold:
            break

        col_to_drop = upper.max().idxmax()
        print(f"Dropping {col_to_drop}: correlation={max_corr:.2f}")
        data = data.drop(columns=[col_to_drop])

    return data

# 3. Apply the filter
X_filtered = drop_highly_correlated(X, threshold=0.5)

print("\nFinal Selected Variables:")
print(X_filtered.columns.tolist())

Dropping bio16_precipitation_wettest_quarter: correlation=0.99
Dropping bio6_min_temp_coldest_month: correlation=0.95
Dropping bio7_temperature_annual_range: correlation=0.94
Dropping bio5_max_temp_warmest_month: correlation=0.94
Dropping bio13_precipitation_wettest_month: correlation=0.93
Dropping bio17_precipitation_driest_quarter: correlation=0.93
Dropping bio4_temperature_seasonality: correlation=0.92
Dropping bio8_mean_temp_wettest_quarter: correlation=0.83
Dropping bio10_mean_temp_warmest_quarter: correlation=0.77
Dropping bio11_mean_temp_coldest_quarter: correlation=0.76
Dropping bio9_mean_temp_driest_quarter: correlation=0.72
Dropping bio3_isothermality: correlation=0.68
Dropping bio2_mean_diurnal_range: correlation=0.65
Dropping bio19_precipitation_coldest_quarter: correlation=0.64
Dropping bio15_precipitation_seasonality: correlation=0.63
Dropping dist_in_km_river: correlation=0.51

Final Selected Variables:
['dist_in_km_railways', 'dist_in_km_road', 'distance_in_km_urban', '

In [4]:
corr = df.drop(columns="presence").corr().round(2)

                                     decimalLongitude  decimalLatitude  \
decimalLongitude                                 1.00             0.33   
decimalLatitude                                  0.33             1.00   
dist_in_km_railways                              0.24            -0.06   
dist_in_km_road                                  0.08             0.05   
distance_in_km_urban                             0.21             0.14   
distance_in_km_lakes                             0.36             0.21   
dist_in_km_river                                 0.04             0.04   
bio1_annual_mean_temperature                    -0.19            -0.31   
bio10_mean_temp_warmest_quarter                 -0.07             0.28   
bio11_mean_temp_coldest_quarter                 -0.33            -0.81   
bio12_annual_precipitation                      -0.02            -0.30   
bio13_precipitation_wettest_month               -0.20            -0.19   
bio14_precipitation_driest_month      

In [5]:
#modelling

In [6]:
#split data

In [7]:
from sklearn.model_selection import train_test_split

# 1. Automate initial feature selection
non_predictors = ['presence', 'koppen_zone', 'decimalLongitude', 'decimalLatitude']
initial_predictors = [col for col in df.columns if col not in non_predictors]

# 2. Apply your correlation filter to the dynamic list
X_filtered = drop_highly_correlated(df[initial_predictors], threshold=0.5)

# 3. Automatically extract the surviving feature names
final_predictors = X_filtered.columns.tolist()
print(f"Selected {len(final_predictors)} predictors:\n{final_predictors}")

# 4. Assign final X and y
X = df[final_predictors]
y = df['presence']

# 5. Split the data using the filtered features
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

Dropping bio16_precipitation_wettest_quarter: correlation=0.99
Dropping bio6_min_temp_coldest_month: correlation=0.95
Dropping bio7_temperature_annual_range: correlation=0.94
Dropping bio5_max_temp_warmest_month: correlation=0.94
Dropping bio13_precipitation_wettest_month: correlation=0.93
Dropping bio17_precipitation_driest_quarter: correlation=0.93
Dropping bio4_temperature_seasonality: correlation=0.92
Dropping bio8_mean_temp_wettest_quarter: correlation=0.83
Dropping bio10_mean_temp_warmest_quarter: correlation=0.77
Dropping bio11_mean_temp_coldest_quarter: correlation=0.76
Dropping bio9_mean_temp_driest_quarter: correlation=0.72
Dropping bio3_isothermality: correlation=0.68
Dropping bio2_mean_diurnal_range: correlation=0.65
Dropping bio19_precipitation_coldest_quarter: correlation=0.64
Dropping bio15_precipitation_seasonality: correlation=0.63
Dropping dist_in_km_river: correlation=0.51
Selected 8 predictors:
['dist_in_km_railways', 'dist_in_km_road', 'distance_in_km_urban', 'dist

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf_params = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

rf_grid.fit(X_train_scaled, y_train)

rf_best = rf_grid.best_estimator_

print(rf_grid.best_params_)

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500}


In [10]:
from sklearn.tree import DecisionTreeClassifier

dt_params = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

dt_grid.fit(X_train_scaled, y_train)

dt_best = dt_grid.best_estimator_
print("Best Parameters:", dt_grid.best_params_)

C:\Users\hibah\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 5}


In [11]:
from sklearn.neighbors import KNeighborsClassifier

knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    knn_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

knn_grid.fit(X_train_scaled, y_train)

knn_best = knn_grid.best_estimator_
print("Best Parameters:", knn_grid.best_params_)

Best Parameters: {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}


In [12]:
from sklearn.linear_model import LogisticRegression

lr_params = {
    'C': [0.01, 0.1, 1, 10, 100]
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=5000),
    lr_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

lr_grid.fit(X_train_scaled, y_train)

lr_best = lr_grid.best_estimator_
print("Best Parameters:", lr_grid.best_params_)

Best Parameters: {'C': 0.1}


In [13]:
from xgboost import XGBClassifier

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1]
}

xgb_grid = GridSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ),
    xgb_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

xgb_grid.fit(X_train_scaled, y_train)

xgb_best = xgb_grid.best_estimator_
print("Best Parameters:", xgb_grid.best_params_)

Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

svm_params = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}

svm_grid = GridSearchCV(
    SVC(probability=True),
    svm_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

svm_best = svm_grid.best_estimator_

print("Best Parameters:", svm_grid.best_params_)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        auc = None

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": auc
    }

In [ ]:
results = pd.DataFrame({
    "Random Forest": evaluate_model(rf_best, X_test_scaled, y_test),
    "Decision Tree": evaluate_model(dt_best, X_test_scaled, y_test),
    "KNN": evaluate_model(knn_best, X_test_scaled, y_test),
    "Logistic": evaluate_model(lr_best, X_test_scaled, y_test),
    "XGBoost": evaluate_model(xgb_best, X_test_scaled, y_test),
    "SVM": evaluate_model(svm_best, X_test_scaled, y_test)
}).T